# 03a - Baseline Cohort Construction

Short summary:
[03a - Baseline Cohort Construction (SHORT)](../docs/03a_baseline_SHORT.md)

---


<!-- Generated notebook input context -->

## Inputs and execution mode

Input definitions: [BigQuery mimic-iv-portfolio.copd_raas.cohort_copd_baseline; 03_build_baseline.sql](../sql/03_build_baseline.sql).

New execution uses the queries and shared loader shown in the code. The stored outputs below were refreshed from saved clinical aggregates, without a new clinical query or model refit.

Stored-output evidence: [original clinical execution](../docs/NOTEBOOK_EXECUTION_VALIDATION.json) · [presentation refresh](../docs/NOTEBOOK_PRESENTATION_VALIDATION.json).

## 1. Objective
Construct the baseline COPD ICU cohort with demographic characteristics, pre-ICU RAAS exposure, and core cohort eligibility fields.

Preserve the admission-age estimate from phase 01, including the raw anchor-age and oldest-group flag; do not replace it with `anchor_age`. This baseline assembly does not add first-day SOFA.

---

## 2. Data Processing Steps


### 2.1 Common BigQuery Utilities


In [2]:
# Shared ADC client with US location, dry runs and per-job byte guards.
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from notebook_bq_runtime import client_for_notebook, run_sql_script, query_to_df
client = client_for_notebook()
print("Bounded BigQuery helpers configured for mimic-iv-portfolio (US); see the execution or presentation record for input origin.")
import notebook_presentation as presentation


Bounded BigQuery helpers configured for mimic-iv-portfolio (US); see the execution or presentation record for input origin.


### 2.2 Execute SQL to Construct the Baseline COPD ICU Cohort


In [3]:
# Execute SQL script (03_build_baseline.sql) to create the baseline table (copd_raas.cohort_copd_baseline) for downstream modeling
run_sql_script("../sql/03_build_baseline.sql")

Presentation replay of saved construction evidence: 03_build_baseline.sql (SQL not executed)


### 2.3 Sanity Checks on Cohort Size and Exposure Distribution


In [4]:
# Verify updated baseline table
# Row-level data are loaded for schema validation only; no rows are displayed.
df_baseline = query_to_df("""
    SELECT *
    FROM `mimic-iv-portfolio.copd_raas.cohort_copd_baseline`
""")

df_baseline.info()
dimensions = presentation.dimensions_frame(df_baseline.shape)
presentation.display_table("03a", "dimensions", dimensions, presentation.dimensions_summary(dimensions))
age_statistics = presentation.age_statistics_frame(df_baseline["age"].describe())
presentation.display_table("03a", "age", presentation.age_table(age_statistics), presentation.age_summary(age_statistics))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11964 entries, 0 to 11963
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   subject_id           11964 non-null  Int64         
 1   hadm_id              11964 non-null  Int64         
 2   stay_id              11964 non-null  Int64         
 3   intime               11964 non-null  datetime64[us]
 4   outtime              11962 non-null  datetime64[us]
 5   icu_los              11962 non-null  float64       
 6   age                  11964 non-null  Int64         
 7   anchor_age           11964 non-null  Int64         
 8   age_anchor_topcoded  11964 non-null  Int64         
 9   gender               11964 non-null  object        
 10  anchor_year          11964 non-null  Int64         
 11  anchor_year_group    11964 non-null  object        
 12  hosp_los             11964 non-null  float64       
 13  raas_pre_icu         11964 non-

Measure,Value
Rows,11964
Columns,15


,Statistic,Value,Unit
0,count,"11,964",admissions
1,mean,71.3,years
2,std,11.4,years
3,min,19.0,years
4,25%,64.0,years
5,50%,72.0,years
6,75%,80.0,years
7,max,102.0,years


In [5]:
# Check distribution of raas_pre_icu without loading row-level data.
raas_distribution = query_to_df("""
    SELECT
      raas_pre_icu,
      COUNT(*) AS count
    FROM `mimic-iv-portfolio.copd_raas.cohort_copd_baseline`
    GROUP BY raas_pre_icu
    ORDER BY raas_pre_icu
""")
binary_table, binary_summary = presentation.binary_orders(raas_distribution, count="count")
presentation.display_table("03a", "binary", binary_table, binary_summary)

Prescription-order group,Admissions
No qualifying RAAS orders,11018
Qualifying RAAS orders,946


The table values and numerical summaries are generated from the selected aggregate results. Age remains approximate.

---

## 3. Output Dataset

**Input Dataset**
- `mimic-iv-portfolio.copd_raas.cohort_copd`
- `mimic-iv-portfolio.copd_raas.cohort_copd_raas`
- MIMIC-IV patient and admission tables in BigQuery.

**Output Dataset**
- `mimic-iv-portfolio.copd_raas.cohort_copd_baseline`

**Downstream Usage**
- This baseline cohort feeds `sql/04_build_outcomes.sql`, followed by `sql/04_build_outcomes_extended.sql`. Complete both scripts after phase 03 and before 04a.
- The shared 04a–04d loader reads the extended outcome table and detailed order flags; the notebooks fit models rather than create these outcome tables. See [the dependency map](../docs/METHODS_OVERVIEW.md#outcome-table-dependencies).


---

## Next Step

Proceed to:
- [03b - Merge COPD Cohort with RAAS Exposures](03b_merge_exposures.ipynb)

This next notebook merges detailed ACE inhibitor and ARB exposure flags into the COPD cohort.
